# Analytics across multiple cases

1. Read all directories (projects) in a data directory
2. Upload each all text files in each project to the Pinecone index
3. Read Pinecone index
4. For a query, find relevant documents
5. Using Langchain, send the query and relevant documents to ChatGPT
6. Get the answer

In [1]:
import openai
import os
from langchain.document_loaders import DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings.openai import OpenAIEmbeddings
from langchain.vectorstores import Pinecone as Pinecone_lch
from langchain.llms import OpenAI
from langchain.chains.question_answering import load_qa_chain
from casify_pinecone_lch import get_answer
from dotenv import load_dotenv, find_dotenv
import pinecone
# from casify_pinecone_lch import index_name
# from casify_pinecone_lch import MODEL
from langchain.document_loaders import TextLoader
from store_document import do_store_document

/home/mark/anaconda3/envs/CasifyAI/lib/python3.9/site-packages/pinecone/index.py:4: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm
/home/mark/anaconda3/envs/CasifyAI/lib/python3.9/site-packages/langchain/llms/openai.py:202: UserWarning: You are trying to use a chat model. This way of initializing it is no longer supported. Instead, please use: `from langchain.chat_models import ChatOpenAI`
  warnings.warn(
/home/mark/anaconda3/envs/CasifyAI/lib/python3.9/site-packages/langchain/llms/openai.py:789: UserWarning: You are trying to use a chat model. This way of initializing it is no longer supported. Instead, please use: `from langchain.chat_models import ChatOpenAI`
  warnings.warn(


In [2]:
_ = load_dotenv(find_dotenv()) # read local .env file
openai.api_key  = os.getenv('OPENAI_API_KEY')

In [3]:
pinecone.init(
    api_key=os.getenv('PINECONE_API_KEY'),     
    environment=os.getenv('PINECONE_ENVIRONMENT')
)
index_name = os.getenv('PINECONE_INDEX_NAME')
pinecone.describe_index(index_name)
index_pc = pinecone.Index(index_name)
print(str(index_pc))
description = pinecone.describe_index(index_name)
print(description)

IndexDescription(name='casify-test', metric='cosine', replicas=1, dimension=1536.0, shards=1, pods=1, pod_type='p1.x1', status={'ready': True, 'state': 'Ready'}, metadata_config=None, source_collection='')


In [4]:
def split_docs(documents, chunk_size=1000, chunk_overlap=20):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    splits = text_splitter.split_documents(documents)
    return splits

In [5]:
def load_docs(directory):
  loader = DirectoryLoader(directory)
  documents = loader.load()
  return documents

In [6]:
from pathlib import Path

# specify the path to the directory. Use Path('.') for current directory or specify a path like Path('/path/to/directory')
data_dir = "../data/Police_reports"
parent_directory = Path(data_dir)

# list all subdirectories
iter_count_for_print = 0
for item in parent_directory.iterdir():
    if item.is_dir():
        case_dir_name = item.name        
        case_id = case_dir_name.split(' ')[0]        
        namespace = case_id
        
        case_dir_path = Path(data_dir + "/" + case_dir_name)               
        documents = load_docs(case_dir_path)
        splits = split_docs(documents)        
        embeddings = OpenAIEmbeddings(openai_api_key=openai.api_key)
        
        index_pc.delete(deleteAll = True, namespace = namespace)
        Pinecone_lch.from_documents(splits, embeddings, index_name=index_name, namespace=namespace)
        #pinecone.describe_index(index_name)
        #index_pc = pinecone.Index("casify")
        #description = pinecone.describe_index(index_name)
        #print(description)
        iter_count_for_print += 1
        print("Researching: " + str(iter_count_for_print), flush=True)        
        query = "Was there murder or injury?"
        answer=get_answer(query, namespace)
        if (answer.lower().startswith("yes") or iter_count_for_print <=5):            
            print("case_dir_name: " + case_dir_name)
            print("case_dir_path: " + str(case_dir_path))
            print("splits len: " + str(len(splits)))
            print("index_name: " + index_name)
            print("namespace: " + namespace)           
            print(answer)

Researching: 1
case_dir_name: 15N01340 Holly M. Woesner
case_dir_path: ../data/Police_reports/15N01340 Holly M. Woesner
splits len: 15
index_name: casify-test
namespace: 15N01340
No, there was no murder or injury mentioned in the context. The crime involved was selling alcohol to a minor.
Researching: 2
case_dir_name: 13H03591 Holly M. Woesner
case_dir_path: ../data/Police_reports/13H03591 Holly M. Woesner
splits len: 2
index_name: casify-test
namespace: 13H03591
The text does not provide information about a murder or injury.
Researching: 3
case_dir_name: 15F07864 Holly M. Woesner
case_dir_path: ../data/Police_reports/15F07864 Holly M. Woesner
splits len: 20
index_name: casify-test
namespace: 15F07864
No, there was no mention of murder or injury in the provided context.
Researching: 4
case_dir_name: 12N07812 Holly M. Woesner
case_dir_path: ../data/Police_reports/12N07812 Holly M. Woesner
splits len: 3
index_name: casify-test
namespace: 12N07812
No, there was no mention of murder or inj

In [ ]:
#loader = TextLoader("../data/159/processed/pages/courtdoc_page_1.txt")
#document = loader.load()

In [ ]:
#documents = [document]

In [ ]:
#docs = split_docs(documents)

In [ ]:
Pinecone_lch.from_documents(docs, embeddings, index_name=index_name, namespace=namespace)

In [ ]:
from langchain.document_loaders import DirectoryLoader

directory = pages_dir

def load_docs(directory):
  loader = DirectoryLoader(directory)
  documents = loader.load()
  return documents

documents = load_docs(directory)
len(documents)

https://python.langchain.com/en/latest/modules/indexes/text_splitters/getting_started.html

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

def split_docs(documents,chunk_size=1000,chunk_overlap=20):
  text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
  docs = text_splitter.split_documents(documents)
  return docs

docs = split_docs(documents)
print(len(docs))

### Print example

In [ ]:
print(docs[0].page_content)
print(len(docs))

In [ ]:
import openai
from langchain.embeddings.openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(openai_api_key=openai.api_key)

query_result = embeddings.embed_query("Hello world")
len(query_result)

https://python.langchain.com/en/latest/modules/indexes/vectorstores/examples/pinecone.html

In [ ]:
# write all data to index_lch
index_lch = Pinecone_lch.from_documents(docs, embeddings, index_name=index_name, namespace = namespace)

# if you already have an index, you can load it like this
index_lch = Pinecone_lch.from_existing_index(index_name, embeddings, namespace = namespace)
index_pc = pinecone.Index(index_name)
index_stats_response = index_pc.describe_index_stats()
print(index_stats_response)

In [ ]:
def get_similar_docs(query,num_sources=5,score=False):
  if score:
    similar_docs = index_lch.similarity_search_with_score(query,k=num_sources)
  else:
    similar_docs = index_lch.similarity_search(query,k=num_sources)
  return similar_docs


In [ ]:
from langchain.llms import OpenAI

llm = OpenAI(model_name=MODEL)

chain = load_qa_chain(llm, chain_type="stuff")

https://python.langchain.com/en/latest/use_cases/question_answering.html

In [ ]:
answer=get_answer("What is the charge?", "159")
print(answer)